### 1. Environment & Setup
Unsloth 'Nuclear' installation and Google Drive mounting.

In [ ]:
# Unsloth "Nuclear" installation script
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-l49f977w/unsloth_f27af9c245e147de837fc2d462fb1d05
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-l49f977w/unsloth_f27af9c245e147de837fc2d462fb1d05
  Resolved https://github.com/unslothai/unsloth.git to commit 8cc05ac89c2c1e15ad1e8fbd1baf7b7e2a5fa463
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 154.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 20.0 MB/s eta 0:00:00
  

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import re

# Base Paths
BASE_PATH = "/content/drive/MyDrive/colab_data/HIPE-2026-data"
AT_ADAPTER = os.path.join(BASE_PATH, "trained_models/qwen_8b_at_adapter")
ISAT_ADAPTER = os.path.join(BASE_PATH, "trained_models/qwen_8b_isAt_adapter")
DATA_PATH = os.path.join(BASE_PATH, "data/sandbox")

LANGUAGES = ['en', 'de', 'fr']

### 2. Specialized Logic Requirements
Robust harvester and data loading utilities.

In [ ]:
def harvest_json_robust(text):
    # Normalize smart quotes
    text = text.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")

    try:
        # Attempt to parse the entire text as a single JSON object
        full_json = json.loads(text)
        # If it's a dictionary with a 'results' key that's a non-empty list
        if isinstance(full_json, dict) and "results" in full_json and \
           isinstance(full_json["results"], list) and len(full_json["results"]) > 0:
            # Return the first item from the 'results' list, wrapped in a list
            # This is to make it compatible with the existing iteration logic `for p in parsed:`
            return [full_json["results"][0]]
        # If it's a dictionary but doesn't have the 'results' structure, return it as is (wrapped in a list)
        elif isinstance(full_json, dict):
            return [full_json]
    except json.JSONDecodeError:
        # If direct parsing fails, proceed to regex-based extraction
        pass

    # Fallback: Extract JSON objects using regex (original logic, for simple cases or partial outputs)
    matches = re.findall(r'\{[^{}]*\}', text)
    results = []
    for match in matches:
        # Correct unquoted labels
        match = re.sub(r':\s*TRUE\b', ': "TRUE"', match, flags=re.IGNORECASE)
        match = re.sub(r':\s*FALSE\b', ': "FALSE"', match, flags=re.IGNORECASE)
        match = re.sub(r':\s*PROBABLE\b', ': "PROBABLE"', match, flags=re.IGNORECASE)
        try:
            results.append(json.loads(match))
        except json.JSONDecodeError:
            pass

    # Return results if any were found, otherwise an empty list to indicate no valid JSON was harvested
    return results if results else []

def load_data(lang):
    filepath = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")
    data = []
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
    else:
        print(f"Warning: {filepath} not found.")
    return data

### 3. Prompt Templates
Defining the prompt templates matching the training setup.

In [ ]:
at_prompt = """You are an expert computational historian specializing in relation extraction.
TASK: Determine the historical relation 'at' between the designated Persons and Places based on the text.

CRITICAL LOGICAL CONSTRAINTS:
1. 'at' represents a permanent, structural, institutional, professional, or residency-based geographic connection over time.
2. Use 'TRUE' ONLY if there is 100% certainty and explicit absolute proof of the connection.
3. Use 'PROBABLE' for 'at' when strong contextual, regional, or family/professional affiliation implies geographic connectivity without explicit absolute proof.
4. Use 'FALSE' if no evidence is present or the context contradicts such a relation.

TARGET TEXT FOR ANALYSIS:
"{text}"

Evaluate the 'at' relationship for the following requested pairs exactly in sequence.
Return a JSON object with key "results" containing one object per pair: {{"at": "TRUE/PROBABLE/FALSE"}}.


PAIRS TO EVALUATE:
{pairs_list_str}"""

isAt_prompt = """You are an expert computational historian specializing in relation extraction.
TASK: Determine the temporal relation 'isAt' between the designated Persons and Places based on the text.

CRITICAL LOGICAL CONSTRAINTS:
1. 'isAt' represents literal immediate physical presence at that place within the narrative moment (the temporal horizon of the article).
2. 'isAt' is TRUE if there is evidence the person was at the location up to about one month before the publication date.
3. Use 'FALSE' if the person is elsewhere, the event happened in the distant past, or no evidence of current presence exists.

TARGET TEXT FOR ANALYSIS:
"{text}"

Evaluate the 'isAt' relationship for the following requested pairs exactly in sequence.
Return a JSON object with key "results" containing one object per pair: {{"isAt": "TRUE/FALSE"}}.


PAIRS TO EVALUATE:
{pairs_list_str}"""

def format_chat_prompt(relation, person, place, text):
    pairs_list_str = f"Person: {person}, Place: {place}"
    if relation == 'at':
        user_msg = at_prompt.format(text=text, pairs_list_str=pairs_list_str)
    else:
        user_msg = isAt_prompt.format(text=text, pairs_list_str=pairs_list_str)

    return [{"role": "user", "content": user_msg}]

### 4. Sequential Inference
Loading base model, `at` inference, followed by adapter switch to `isAt` and Logic Guard.

In [ ]:
from unsloth import FastLanguageModel
import torch
import sys
import os

max_seq_length = 4096

if not os.path.exists(AT_ADAPTER):
    print(f"Error: AT adapter path does not exist: {AT_ADAPTER}")
    sys.exit(1)

# Load base model
print("Loading Base Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit", # Using the exact name requested
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

Loading Base Model...
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/Qwen3-8B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
from tqdm.notebook import tqdm

# 1. AT Inference
print(f"Loading AT-Specialist from {AT_ADAPTER}...")
if "at_adapter" not in getattr(model, "peft_config", {}):
    model.load_adapter(AT_ADAPTER, adapter_name="at_adapter")
model.set_adapter("at_adapter")
FastLanguageModel.for_inference(model)

run_lang = "fr"

if 'at_predictions' not in locals():
    at_predictions = {}
at_predictions[run_lang] = []

print(f"Running AT inference for {run_lang.upper()}...")
data = load_data(run_lang)

for item in tqdm(data, desc="Processing Documents (AT)"):
    for pair in item.get('sampled_pairs', []):
        pers_list = pair.get('pers_mentions_list', [])
        loc_list = pair.get('loc_mentions_list', [])
        person = pers_list[0] if pers_list else ""
        place = loc_list[0] if loc_list else ""

        messages = format_chat_prompt('at', person, place, item.get('text', ''))
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

        max_retries = 3
        for attempt in range(max_retries):
            # Kept simple without stop strings
            outputs = model.generate(input_ids=inputs, max_new_tokens=2048, do_sample=False)
            response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

            resp_clean = response.strip().upper()
            if resp_clean in ['TRUE', 'FALSE', 'PROBABLE']:
                pred = resp_clean
            else:
                parsed = harvest_json_robust(response)
                pred = 'ERROR'
                for p in parsed:
                    if isinstance(p, dict):
                        for val in p.values():
                            if str(val).upper() in ['TRUE', 'FALSE', 'PROBABLE']:
                                pred = str(val).upper()
                                break
                    if pred != 'ERROR':
                        break

                if pred == 'ERROR':
                    if 'PROBABLE' in resp_clean:
                        pred = 'PROBABLE'
                    elif 'TRUE' in resp_clean:
                        pred = 'TRUE'
                    elif 'FALSE' in resp_clean:
                        pred = 'FALSE'

            if pred != 'ERROR':
                break
            elif attempt < max_retries - 1:
                print(f"  [AT Retry {attempt+1}] Model returned error, re-prompting...")

        if pred == 'ERROR':
            print(f"[AT ERROR] doc: {item.get('document_id')} | pair: {person}-{place} | Raw Response: {response}")
            pred = 'FALSE' # Safety net to prevent schema failure

        pair['at'] = pred

    at_predictions[run_lang].append(item)

Loading AT-Specialist from /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/qwen_8b_at_adapter...


Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Running AT inference for FR...


Processing Documents (AT):   0%|          | 0/107 [00:00<?, ?it/s]

Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [AT Retry 2] Model returned error, re-prompting...


Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[AT ERROR] doc: train_fr_8 | pair: Edouard Daladier-Seine-et-Oise | Raw Response: <think>
Okay, let's tackle this query. The user wants to know if the person "Edouard Daladier" is "at" the place "Seine-et-Oise" based on the provided text.

First, I'll scan through the text for any mentions of Edouard Daladier and Seine-et-Oise. 

Looking at the section about the "RACCOURCIS" part, there's a mention of "Il y a au moins un journaliste français qui a le sens de l'humour et de l'actualité : c'est notre bon ami Gustave Hervé." But that doesn't seem related. 

Wait, further down under the "LES GRANDS REPORTAGES" section, there's a line: "Il y a au moins un journaliste français qui a le sens de l'humour et de l'actualité : c'est notre bon ami Gustave Hervé." Still no mention of Seine-et-Oise.

Wait, maybe I missed something. Let me check again. The text mentions "Seine-et-Oise" in the context of a police band: "Il y a au moins un journaliste français qui a le sens de l'humour et de l'actualit

Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

In [ ]:
import torch
import gc

# Delete the model and trainer from memory
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Now load the base model fresh
print("Loading Base Model for ISAT...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

# Load the isAt adapter
print(f"Loading ISAT-Specialist from {ISAT_ADAPTER}...")
model.load_adapter(ISAT_ADAPTER, adapter_name="isAt_adapter")
model.set_adapter("isAt_adapter")
FastLanguageModel.for_inference(model)

print("Ready to run ISAT inference loop.")


Loading Base Model for ISAT...
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/Qwen3-8B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loading ISAT-Specialist from /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/qwen_8b_isAt_adapter...


Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Ready to run ISAT inference loop.


In [ ]:
from tqdm.notebook import tqdm
import time
import json
import os

# 2. ISAT Inference
run_lang = "fr"

if 'final_results' not in locals():
    final_results = {}
final_results[run_lang] = []

print(f"Running ISAT inference for {run_lang.upper()}...")

if run_lang not in at_predictions or not at_predictions[run_lang]:
    print(f"Error: You must run the AT Inference block for '{run_lang}' first!")
else:
    total_docs = len(at_predictions[run_lang])
    for doc_idx, item in enumerate(tqdm(at_predictions[run_lang], desc="Processing Documents (ISAT)")):
        doc_id = item.get('document_id', 'Unknown')
        pairs = item.get('sampled_pairs', [])


        for i, pair in enumerate(pairs):
            pers_list = pair.get('pers_mentions_list', [])
            loc_list = pair.get('loc_mentions_list', [])
            person = pers_list[0] if pers_list else ""
            place = loc_list[0] if loc_list else ""

            messages = format_chat_prompt('isAt', person, place, item.get('text', ''))
            inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

            max_retries = 3
            for attempt in range(max_retries):
                start_time = time.time()
                # Kept simple without stop strings
                outputs = model.generate(input_ids=inputs, max_new_tokens=2048, do_sample=False, use_cache=True)
                end_time = time.time()

                gen_duration = end_time - start_time
                response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

                resp_clean = response.strip().upper()
                if resp_clean in ['TRUE', 'FALSE']:
                    isAt_pred = resp_clean
                else:
                    parsed = harvest_json_robust(response)
                    isAt_pred = 'ERROR'
                    for p in parsed:
                        if isinstance(p, dict):
                            for val in p.values():
                                if str(val).upper() in ['TRUE', 'FALSE']:
                                    isAt_pred = str(val).upper()
                                    break
                        if isAt_pred != 'ERROR':
                            break

                if isAt_pred == 'ERROR':
                    if 'TRUE' in resp_clean:
                        isAt_pred = 'TRUE'
                    elif 'FALSE' in resp_clean:
                        isAt_pred = 'FALSE'

                if isAt_pred != 'ERROR':
                    break
                elif attempt < max_retries - 1:
                    print(f"    [ISAT Retry {attempt+1}] Model returned error, re-prompting...")

            if isAt_pred == 'ERROR':
                print(f"[isAt ERROR] doc: {doc_id} | pair: {person}-{place} | Raw Response: {response}")
                isAt_pred = 'FALSE' # Safety net to prevent schema failure

            pair['isAt'] = isAt_pred

        final_results[run_lang].append(item)

    out_path = os.path.join(BASE_PATH, f"integrated_qwen_8b_{run_lang}_results.jsonl")
    with open(out_path, 'w', encoding='utf-8') as f:
        for res in final_results[run_lang]:
            f.write(json.dumps(res) + '\n')
    print(f"\nDONE. Saved final predictions to {out_path}")

Running ISAT inference for FR...


Processing Documents (ISAT):   0%|          | 0/107 [00:00<?, ?it/s]

Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

    [ISAT Retry 1] Model returned error, re-prompting...


Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

    [ISAT Retry 2] Model returned error, re-prompting...


Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[isAt ERROR] doc: hidden_test_fr_22 | pair: Riri-passage Ganne¬
ron | Raw Response: 不远的将来，我们将在巴黎的大型百货公司杜瓦耶 (Dufayel) 参观展览，展示成千上万的家具、座位、地毯、壁挂、壁炉、照明设备、家务用品、床单、旅行用品和运动器材。音乐会、电影和下午茶。 (通讯) 外国人在我们的大型餐厅里享受着乐趣，如 Zang 家的著名面包，这是这家店的创新，受到了许多模仿。 (通讯) 《MATIN》的《「MATIN」》 1910 年 1 月 15 日的「大团圆」 由 Léon Sazie 创作的全新小说 第一章 不可见的老师 XXVIII 为了战胜 Riri ... 但 Riri 没有意识到，一个工友在她面前已经看到了这一切，整个场景。 自然，她和其他工友谈论过。 在年轻女孩之间讨论过 Riri 的行为。 但 Riri 的行为，自然，所有的工友都认识 Riri，她是 La Guairière 公爵，也就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，他就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，他就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，他就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，他就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，他就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，他就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，他就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，他就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，他就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，他就是所谓的贵族。 他就是那个在 Riri 的地方，也就是在巴黎的 La Guairière 公爵，

Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


DONE. Saved final predictions to /content/drive/MyDrive/colab_data/HIPE-2026-data/integrated_qwen_8b_fr_results.jsonl


### 5. Evaluation
Automatically trigger the official scorer scripts for each integrated output file.

In [ ]:
import json
import os

print("Running automated evaluation script...")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_qwen_8b_{lang}_results.jsonl")
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if os.path.exists(pred_path) and os.path.exists(gold_path):
        print(f"\nEvaluating {lang}...")
        # cd into BASE_PATH so the script can find the 'schemas/' directory
        !cd "{BASE_PATH}" && python scripts/file_scorer_evaluation.py --predictions_file "{pred_path}" --gold_data_file "{gold_path}"
    else:
        print(f"Skipping eval for {lang}. Check if prediction or gold files exist.")

Running automated evaluation script...

Evaluating en...

Evaluation Results for integrated_qwen_8b_en_results.jsonl:
  'at': macro_recall=0.5652, accuracy=0.4967 (75/151)
  'isAt': macro_recall=0.7030, accuracy=0.4768 (72/151)
  'global': macro_recall=0.6341 (147/302)


Evaluating de...

Evaluation Results for integrated_qwen_8b_de_results.jsonl:
  'at': macro_recall=0.5697, accuracy=0.5023 (216/432)
  'isAt': macro_recall=0.7484, accuracy=0.5903 (255/432)
  'global': macro_recall=0.6591 (471/864)


Evaluating fr...

Evaluation Results for integrated_qwen_8b_fr_results.jsonl:
  'at': macro_recall=0.5850, accuracy=0.5968 (894/1498)
  'isAt': macro_recall=0.7772, accuracy=0.6575 (985/1498)
  'global': macro_recall=0.6811 (1879/2996)



In [ ]:
import json
import os

print("Detailed Prediction Analysis by Label")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_qwen_8b_{lang}_results.jsonl")
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if not (os.path.exists(pred_path) and os.path.exists(gold_path)):
        print(f"Missing files for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Analysis for {lang.upper()} ---")
    print(f"{'='*40}")

    # 1. Load gold data into a dictionary mapped by document_id
    gold_data = {}
    with open(gold_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            doc_id = item['document_id']
            # Map pair entities to their gold pairs so order doesn't matter
            gold_data[doc_id] = {}
            for pair in item.get('sampled_pairs', []):
                pers_id = pair.get('pers_entity_id')
                loc_id = pair.get('loc_entity_id')
                gold_data[doc_id][(pers_id, loc_id)] = pair

    # 2. Track stats
    at_stats = {
        "TRUE": {"correct": 0, "wrong": 0},
        "FALSE": {"correct": 0, "wrong": 0},
        "PROBABLE": {"correct": 0, "wrong": 0}
    }
    isat_stats = {
        "TRUE": {"correct": 0, "wrong": 0},
        "FALSE": {"correct": 0, "wrong": 0}
    }

    # 3. Compare predictions against gold
    with open(pred_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            doc_id = item['document_id']
            pred_pairs = item.get('sampled_pairs', [])

            for p_pair in pred_pairs:
                pers_id = p_pair.get('pers_entity_id')
                loc_id = p_pair.get('loc_entity_id')

                # Find corresponding gold pair
                g_pair = gold_data.get(doc_id, {}).get((pers_id, loc_id))

                if not g_pair:
                    continue # Skip if no matching gold pair found

                # --- Check 'at' Field ---
                g_at = g_pair.get('at', 'FALSE')
                p_at = p_pair.get('at', 'ERROR')

                if g_at in at_stats:
                    if g_at == p_at:
                        at_stats[g_at]['correct'] += 1
                    else:
                        at_stats[g_at]['wrong'] += 1

                # --- Check 'isAt' Field ---
                g_isat = g_pair.get('isAt', 'FALSE')
                p_isat = p_pair.get('isAt', 'ERROR')

                if g_isat in isat_stats:
                    if g_isat == p_isat:
                        isat_stats[g_isat]['correct'] += 1
                    else:
                        isat_stats[g_isat]['wrong'] += 1

    # 4. Print Results
    print("\n[ 'AT' FIELD STATS ]")
    for label, counts in at_stats.items():
        total = counts['correct'] + counts['wrong']
        acc = (counts['correct'] / total * 100) if total > 0 else 0
        print(f"  Gold={label:<8}: {counts['correct']:>4} Correct | {counts['wrong']:>4} Wrong | Recall: {acc:>5.1f}%")

    print("\n[ 'ISAT' FIELD STATS ]")
    for label, counts in isat_stats.items():
        total = counts['correct'] + counts['wrong']
        acc = (counts['correct'] / total * 100) if total > 0 else 0
        print(f"  Gold={label:<8}: {counts['correct']:>4} Correct | {counts['wrong']:>4} Wrong | Recall: {acc:>5.1f}%")


Detailed Prediction Analysis by Label

--- Analysis for EN ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :   29 Correct |    0 Wrong | Recall: 100.0%
  Gold=FALSE   :   41 Correct |   27 Wrong | Recall:  60.3%
  Gold=PROBABLE:    5 Correct |   49 Wrong | Recall:   9.3%

[ 'ISAT' FIELD STATS ]
  Gold=TRUE    :   18 Correct |    0 Wrong | Recall: 100.0%
  Gold=FALSE   :   54 Correct |   79 Wrong | Recall:  40.6%

--- Analysis for DE ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :   38 Correct |    3 Wrong | Recall:  92.7%
  Gold=FALSE   :  161 Correct |   83 Wrong | Recall:  66.0%
  Gold=PROBABLE:   18 Correct |  129 Wrong | Recall:  12.2%

[ 'ISAT' FIELD STATS ]
  Gold=TRUE    :   27 Correct |    2 Wrong | Recall:  93.1%
  Gold=FALSE   :  228 Correct |  175 Wrong | Recall:  56.6%

--- Analysis for FR ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :  154 Correct |   25 Wrong | Recall:  86.0%
  Gold=FALSE   :  670 Correct |  282 Wrong | Recall:  70.4%
  Gold=PROBABLE:   70 Correct |  297 Wrong | Recall: 

In [ ]:
import json
import os

print("Prediction Distribution (Model Guesses)")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_qwen_8b_{lang}_results.jsonl")

    if not os.path.exists(pred_path):
        print(f"Missing prediction file for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Model Guesses for {lang.upper()} ---")
    print(f"{'='*40}")

    at_guesses = {"TRUE": 0, "FALSE": 0, "PROBABLE": 0, "ERROR": 0}
    isat_guesses = {"TRUE": 0, "FALSE": 0, "ERROR": 0}

    with open(pred_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            for pair in item.get('sampled_pairs', []):
                p_at = pair.get('at', 'ERROR')
                p_isat = pair.get('isAt', 'ERROR')

                if p_at in at_guesses:
                    at_guesses[p_at] += 1
                else:
                    at_guesses['ERROR'] += 1

                if p_isat in isat_guesses:
                    isat_guesses[p_isat] += 1
                else:
                    isat_guesses['ERROR'] += 1

    print("\n[ 'AT' FIELD GUESSES ]")
    for label, count in at_guesses.items():
        if count > 0 or label in ["TRUE", "FALSE", "PROBABLE"]:
            print(f"  Guessed {label:<8}: {count:>4} times")

    print("\n[ 'ISAT' FIELD GUESSES ]")
    for label, count in isat_guesses.items():
        if count > 0 or label in ["TRUE", "FALSE"]:
            print(f"  Guessed {label:<8}: {count:>4} times")

Prediction Distribution (Model Guesses)

--- Model Guesses for EN ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :   94 times
  Guessed FALSE   :   48 times
  Guessed PROBABLE:    9 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :   97 times
  Guessed FALSE   :   54 times

--- Model Guesses for DE ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :  182 times
  Guessed FALSE   :  201 times
  Guessed PROBABLE:   49 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :  202 times
  Guessed FALSE   :  230 times

--- Model Guesses for FR ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :  526 times
  Guessed FALSE   :  774 times
  Guessed PROBABLE:  198 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :  620 times
  Guessed FALSE   :  878 times


In [ ]:
import json
import os

print("Gold Label Distribution (Actual Data)")

for lang in LANGUAGES:
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if not os.path.exists(gold_path):
        print(f"Missing gold file for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Gold Labels for {lang.upper()} ---")
    print(f"{'='*40}")

    at_counts = {"TRUE": 0, "FALSE": 0, "PROBABLE": 0, "ERROR": 0}
    isat_counts = {"TRUE": 0, "FALSE": 0, "ERROR": 0}

    with open(gold_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            for pair in item.get('sampled_pairs', []):
                g_at = pair.get('at', 'ERROR')
                g_isat = pair.get('isAt', 'ERROR')

                if g_at in at_counts:
                    at_counts[g_at] += 1
                else:
                    at_counts['ERROR'] += 1

                if g_isat in isat_counts:
                    isat_counts[g_isat] += 1
                else:
                    isat_counts['ERROR'] += 1

    print("\n[ 'AT' FIELD GOLD LABELS ]")
    for label, count in at_counts.items():
        if count > 0 or label in ["TRUE", "FALSE", "PROBABLE"]:
            print(f"  Actual {label:<8}: {count:>4} times")

    print("\n[ 'ISAT' FIELD GOLD LABELS ]")
    for label, count in isat_counts.items():
        if count > 0 or label in ["TRUE", "FALSE"]:
            print(f"  Actual {label:<8}: {count:>4} times")

Gold Label Distribution (Actual Data)

--- Gold Labels for EN ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :   29 times
  Actual FALSE   :   68 times
  Actual PROBABLE:   54 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :   18 times
  Actual FALSE   :  133 times

--- Gold Labels for DE ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :   41 times
  Actual FALSE   :  244 times
  Actual PROBABLE:  147 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :   29 times
  Actual FALSE   :  403 times

--- Gold Labels for FR ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :  179 times
  Actual FALSE   :  952 times
  Actual PROBABLE:  367 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :  127 times
  Actual FALSE   : 1371 times
